In [ ]:
# =====================================================
# CELL-1 : SETUP + DATASET + DUAL SSL AUGMENTATION
# =====================================================

import os
import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# -----------------------------------------------------
# DEVICE
# -----------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# -----------------------------------------------------
# PATHS
# -----------------------------------------------------

TRAIN_PATH = "/kaggle/input/datasets/sabbir4724/training-data"
TEST_PATH  = "/kaggle/input/datasets/sabbir4724/testing-data"

# -----------------------------------------------------
# REPRODUCIBILITY
# -----------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -----------------------------------------------------
# SSL AUGMENTATION VIEW-1
# -----------------------------------------------------

ssl_transform_1 = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8,1.0)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# -----------------------------------------------------
# SSL AUGMENTATION VIEW-2
# -----------------------------------------------------

ssl_transform_2 = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(
        224,
        scale=(0.7,1.0)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# -----------------------------------------------------
# TEST TRANSFORM
# -----------------------------------------------------

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# -----------------------------------------------------
# SSL DATASET
# RETURNS:
# view1, view2, label
# -----------------------------------------------------

class SSLDataset(Dataset):

    def __init__(self, root):

        self.root = root

        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root,d))
        ])

        self.class_to_idx = {
            c:i for i,c in enumerate(self.classes)
        }

        self.images = []
        self.labels = []

        for cls in self.classes:

            folder = os.path.join(root, cls)

            for img in os.listdir(folder):

                img_path = os.path.join(folder, img)

                self.images.append(img_path)
                self.labels.append(
                    self.class_to_idx[cls]
                )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = self.images[idx]

        img = Image.open(
            img_path
        ).convert("RGB")

        view1 = ssl_transform_1(img)
        view2 = ssl_transform_2(img)

        label = self.labels[idx]

        return view1, view2, label


# -----------------------------------------------------
# TEST DATASET
# -----------------------------------------------------

class TestDataset(Dataset):

    def __init__(self, root):

        self.root = root

        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root,d))
        ])

        self.class_to_idx = {
            c:i for i,c in enumerate(self.classes)
        }

        self.images = []
        self.labels = []

        for cls in self.classes:

            folder = os.path.join(root, cls)

            for img in os.listdir(folder):

                img_path = os.path.join(folder, img)

                self.images.append(img_path)
                self.labels.append(
                    self.class_to_idx[cls]
                )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img = Image.open(
            self.images[idx]
        ).convert("RGB")

        img = test_transform(img)

        label = self.labels[idx]

        return img, label


# -----------------------------------------------------
# LOAD DATASETS
# -----------------------------------------------------

train_dataset = SSLDataset(TRAIN_PATH)

test_dataset = TestDataset(TEST_PATH)

# -----------------------------------------------------
# DATALOADERS
# -----------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# -----------------------------------------------------
# INFO
# -----------------------------------------------------

print("\nClasses:")
print(train_dataset.classes)

print("\nClass Mapping:")
print(train_dataset.class_to_idx)

print("\nTrain Images:", len(train_dataset))
print("Test Images :", len(test_dataset))

# -----------------------------------------------------
# SANITY CHECK
# -----------------------------------------------------

v1, v2, y = train_dataset[0]

print("\nSSL View-1 Shape :", v1.shape)
print("SSL View-2 Shape :", v2.shape)
print("Label            :", y)


In [ ]:

# CELL-2 : MobileNetV3 + ECA + Projection Head

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# -----------------------------------------------------
# ECA ATTENTION
# -----------------------------------------------------

class ECALayer(nn.Module):

    def __init__(self, channels, k_size=3):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.conv = nn.Conv1d(
            1,
            1,
            kernel_size=k_size,
            padding=(k_size - 1) // 2,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        y = self.avg_pool(x)

        y = y.squeeze(-1).transpose(-1, -2)

        y = self.conv(y)

        y = self.sigmoid(y)

        y = y.transpose(-1, -2).unsqueeze(-1)

        return x * y.expand_as(x)


# -----------------------------------------------------
# ENCODER
# -----------------------------------------------------

class MobileNetV3_ECA(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "mobilenetv3_large_100",
            pretrained=True,
            num_classes=0
        )

        self.eca = ECALayer(
            channels=960,
            k_size=3
        )

    def forward(self, x):

        # Feature Map
        feat_map = self.backbone.forward_features(x)

        # ECA Attention
        feat_map = self.eca(feat_map)

        # Global Pooling
        feat = F.adaptive_avg_pool2d(
            feat_map,
            1
        )

        feat = feat.view(
            feat.size(0),
            -1
        )

        return feat


# -----------------------------------------------------
# PROJECTION HEAD
# -----------------------------------------------------

class ProjectionHead(nn.Module):

    def __init__(
        self,
        in_dim=960,
        hidden_dim=512,
        out_dim=128
    ):
        super().__init__()

        self.projector = nn.Sequential(

            nn.Linear(
                in_dim,
                hidden_dim
            ),

            nn.BatchNorm1d(
                hidden_dim
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                hidden_dim,
                out_dim
            )
        )

    def forward(self, x):
        return self.projector(x)


# -----------------------------------------------------
# SSL MODEL
# -----------------------------------------------------

class SSLModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = MobileNetV3_ECA()

        self.projector = ProjectionHead()

    def forward(self, x):

        feat = self.encoder(x)

        proj = self.projector(feat)

        return feat, proj


# -----------------------------------------------------
# BUILD MODEL
# -----------------------------------------------------

ssl_model = SSLModel().to(device)

# -----------------------------------------------------
# PARAMETER COUNT
# -----------------------------------------------------

total_params = sum(
    p.numel()
    for p in ssl_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in ssl_model.parameters()
    if p.requires_grad
)

print("\nModel Summary")
print("="*50)

print(
    f"Total Params     : {total_params:,}"
)

print(
    f"Trainable Params : {trainable_params:,}"
)

# -----------------------------------------------------
# SANITY CHECK
# -----------------------------------------------------

sample_v1, sample_v2, _ = next(
    iter(train_loader)
)

sample_v1 = sample_v1.to(device)

with torch.no_grad():

    features, projections = ssl_model(
        sample_v1
    )

print("\nFeature Shape :", features.shape)
print("Projection Shape :", projections.shape)



In [ ]:
# =====================================================
# CELL-3 : SSL PRETRAINING (UPDATED)
# NT-Xent + Loss Curve + Best Model Save
# =====================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# -----------------------------------------------------
# NT-XENT LOSS
# -----------------------------------------------------

class NTXentLoss(nn.Module):

    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):

        batch_size = z1.size(0)

        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        representations = torch.cat(
            [z1, z2],
            dim=0
        )

        similarity_matrix = torch.matmul(
            representations,
            representations.T
        )

        similarity_matrix = (
            similarity_matrix /
            self.temperature
        )

        mask = torch.eye(
            2 * batch_size,
            dtype=torch.bool
        ).to(z1.device)

        similarity_matrix = similarity_matrix.masked_fill(
            mask,
            -1e9
        )

        positives = torch.cat([
            torch.diag(similarity_matrix, batch_size),
            torch.diag(similarity_matrix, -batch_size)
        ])

        numerator = torch.exp(positives)

        denominator = torch.sum(
            torch.exp(similarity_matrix),
            dim=1
        )

        loss = -torch.log(
            numerator / denominator
        )

        return loss.mean()


# -----------------------------------------------------
# OPTIMIZER
# -----------------------------------------------------

optimizer = torch.optim.AdamW(
    ssl_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

criterion = NTXentLoss(
    temperature=0.5
)

# -----------------------------------------------------
# CONFIG
# -----------------------------------------------------

EPOCHS = 300

best_loss = float("inf")

save_path = "ssl_encoder.pth"

ssl_losses = []

# -----------------------------------------------------
# TRAINING
# -----------------------------------------------------

for epoch in range(EPOCHS):

    ssl_model.train()

    running_loss = 0.0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for view1, view2, _ in pbar:

        view1 = view1.to(device)
        view2 = view2.to(device)

        optimizer.zero_grad()

        _, z1 = ssl_model(view1)
        _, z2 = ssl_model(view2)

        loss = criterion(z1, z2)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = (
        running_loss /
        len(train_loader)
    )

    ssl_losses.append(epoch_loss)

    print(
        f"\nEpoch [{epoch+1}/{EPOCHS}] "
        f"SSL Loss: {epoch_loss:.4f}"
    )

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        torch.save(
            ssl_model.state_dict(),
            save_path
        )

        print(
            f"✅ Best model saved "
            f"(Loss={epoch_loss:.4f})"
        )

# -----------------------------------------------------
# LOAD BEST MODEL
# -----------------------------------------------------

ssl_model.load_state_dict(
    torch.load(
        save_path,
        map_location=device
    )
)

print("\n==================================================")
print("SSL PRETRAINING FINISHED")
print("==================================================")
print(f"Best SSL Loss : {best_loss:.4f}")
print(f"Saved Model   : {save_path}")

# -----------------------------------------------------
# LOSS CURVE
# -----------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    range(1, len(ssl_losses)+1),
    ssl_losses,
    marker='o',
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("NT-Xent Loss")
plt.title("SSL Pretraining Loss Curve")

plt.grid(True)

plt.savefig(
    "ssl_loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved Figure : ssl_loss_curve.png")

# -----------------------------------------------------
# FINAL LOSS TABLE
# -----------------------------------------------------

for i, loss in enumerate(ssl_losses):
    print(
        f"Epoch {i+1:02d} : {loss:.4f}"
    )



In [ ]:

# CELL-3.5 : SUPERVISED FINE-TUNING


import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from torch.utils.data import DataLoader

# =====================================================
# OUTPUT FOLDER
# =====================================================

OUTPUT_DIR = "/kaggle/working/output"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print(f"Output Folder: {OUTPUT_DIR}")

# =====================================================
# DATASET
# =====================================================

class ClassificationDataset(torch.utils.data.Dataset):

    def __init__(self, root):

        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ])

        self.images = []
        self.labels = []

        for idx, cls in enumerate(self.classes):

            folder = os.path.join(root, cls)

            for img in os.listdir(folder):

                self.images.append(
                    os.path.join(folder, img)
                )

                self.labels.append(idx)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img = Image.open(
            self.images[idx]
        ).convert("RGB")

        img = test_transform(img)

        label = self.labels[idx]

        return img, label

# =====================================================
# TRAIN DATASET
# =====================================================

train_cls_ds = ClassificationDataset(
    TRAIN_PATH
)

train_cls_loader = DataLoader(
    train_cls_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(
    f"Training Images: {len(train_cls_ds)}"
)

# =====================================================
# MODEL
# =====================================================

class FineTuneModel(nn.Module):

    def __init__(self, ssl_model):

        super().__init__()

        self.encoder = ssl_model.encoder

        self.classifier = nn.Linear(
            960,
            4
        )

    def forward(self, x):

        feat = self.encoder(x)

        out = self.classifier(feat)

        return out

# =====================================================
# LOAD SSL WEIGHTS
# =====================================================

ssl_model.load_state_dict(

    torch.load(
        "ssl_encoder.pth",
        map_location=device
    )

)

print("SSL Weights Loaded")

# =====================================================
# BUILD MODEL
# =====================================================

finetune_model = FineTuneModel(
    ssl_model
).to(device)

# =====================================================
# PARAM COUNT
# =====================================================

total_params = sum(
    p.numel()
    for p in finetune_model.parameters()
)

print(
    f"Total Parameters: {total_params:,}"
)

# =====================================================
# LOSS + OPTIMIZER
# =====================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(

    finetune_model.parameters(),

    lr=1e-4,

    weight_decay=1e-4
)

# =====================================================
# TRAIN CONFIG
# =====================================================

EPOCHS = 30

best_acc = 0

train_losses = []
train_accs = []

# =====================================================
# TRAINING
# =====================================================

for epoch in range(EPOCHS):

    finetune_model.train()

    running_loss = 0

    correct = 0

    total = 0

    pbar = tqdm(
        train_cls_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for imgs, labels in pbar:

        imgs = imgs.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = finetune_model(imgs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        pbar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = (
        running_loss /
        len(train_cls_loader)
    )

    epoch_acc = (
        correct /
        total
    )

    train_losses.append(
        epoch_loss
    )

    train_accs.append(
        epoch_acc
    )

    print(
        f"Epoch {epoch+1:02d}"
        f" | Loss={epoch_loss:.4f}"
        f" | Acc={epoch_acc:.4f}"
    )

    # -----------------------------------------
    # SAVE BEST MODEL
    # -----------------------------------------

    if epoch_acc > best_acc:

        best_acc = epoch_acc

        torch.save(

            finetune_model.encoder.state_dict(),

            os.path.join(
                OUTPUT_DIR,
                "finetuned_encoder.pth"
            )
        )

        print(
            "✅ Best Encoder Saved"
        )

# =====================================================
# TRAIN HISTORY TABLE
# =====================================================

history_df = pd.DataFrame({

    "Epoch":
    list(range(1, EPOCHS+1)),

    "Train_Loss":
    train_losses,

    "Train_Accuracy":
    train_accs

})

history_csv = os.path.join(
    OUTPUT_DIR,
    "finetuning_history.csv"
)

history_df.to_csv(
    history_csv,
    index=False
)

# =====================================================
# LOSS CURVE
# =====================================================

plt.figure(figsize=(8,5))

plt.plot(
    train_losses,
    marker="o"
)

plt.title(
    "Fine-Tuning Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.grid(True)

loss_fig = os.path.join(
    OUTPUT_DIR,
    "finetuning_loss_curve.png"
)

plt.savefig(
    loss_fig,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# =====================================================
# ACC CURVE
# =====================================================

plt.figure(figsize=(8,5))

plt.plot(
    train_accs,
    marker="o"
)

plt.title(
    "Fine-Tuning Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.grid(True)

acc_fig = os.path.join(
    OUTPUT_DIR,
    "finetuning_accuracy_curve.png"
)

plt.savefig(
    acc_fig,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# =====================================================
# FINAL SUMMARY TABLE
# =====================================================

summary_df = pd.DataFrame({

    "Metric":[
        "Training Images",
        "Epochs",
        "Best Accuracy (%)",
        "Total Parameters"
    ],

    "Value":[
        len(train_cls_ds),
        EPOCHS,
        round(best_acc*100,4),
        f"{total_params:,}"
    ]

})

summary_csv = os.path.join(
    OUTPUT_DIR,
    "finetuning_summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)

# =====================================================
# PRINT RESULTS
# =====================================================

print("\n")
print("="*60)
print("FINE-TUNING COMPLETE")
print("="*60)

print(
    f"Best Train Accuracy : {best_acc*100:.4f}%"
)

print(
    f"Total Parameters    : {total_params:,}"
)

print("\nSaved Files:")

print(
    "✓ finetuned_encoder.pth"
)

print(
    "✓ finetuning_history.csv"
)

print(
    "✓ finetuning_summary.csv"
)

print(
    "✓ finetuning_loss_curve.png"
)

print(
    "✓ finetuning_accuracy_curve.png"
)

print("\n")
print("="*60)
print("SUMMARY TABLE")
print("="*60)

display(summary_df)

print("\nOutput Directory:")
print(OUTPUT_DIR)

In [ ]:
# =====================================================
# CELL-5 : FEW-SHOT INFERENCE COMPARISON
# Same Fine-Tuned SSL Encoder (MobileNetV3 + ECA)
# 5-shot: Siamese Network | Matching Network | Relation Network
# Metrics: Accuracy, Precision, Recall, F1  ->  Comparison Table
# =====================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -----------------------------------------------------
# DEVICE / SEED
# -----------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -----------------------------------------------------
# LOAD FINE-TUNED ENCODER (same encoder as CELL-4)
# assumes ssl_model, test_transform, TRAIN_PATH, TEST_PATH
# already exist in the notebook (from CELL-1/2/3.5/4)
# -----------------------------------------------------

CANDIDATE_ENCODER_PATHS = [
    "finetuned_encoder.pth",
    "/kaggle/working/finetuned_encoder.pth",
    "/kaggle/working/output/finetuned_encoder.pth",
]

encoder_path = None
for p in CANDIDATE_ENCODER_PATHS:
    if os.path.isfile(p):
        encoder_path = p
        break

if encoder_path is None:
    raise FileNotFoundError(
        "Could not find 'finetuned_encoder.pth' in any of: "
        f"{CANDIDATE_ENCODER_PATHS}. "
        "Make sure CELL-3.5 (Supervised Fine-Tuning) has been run in THIS "
        "kernel session first — it saves the encoder to "
        "/kaggle/working/output/finetuned_encoder.pth. If your kernel was "
        "restarted, re-run CELL-1, CELL-2, CELL-3 (or load ssl_encoder.pth), "
        "then CELL-3.5 before running this cell."
    )

print(f"Loading fine-tuned encoder from: {encoder_path}")
ssl_model.encoder.load_state_dict(
    torch.load(encoder_path, map_location=device)
)
ssl_model = ssl_model.to(device)
ssl_model.eval()

print("Fine-Tuned Encoder Loaded")

# -----------------------------------------------------
# DATASET (identical to CELL-4)
# -----------------------------------------------------

class FeatureDataset(Dataset):
    def __init__(self, root):
        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ])
        self.images = []
        self.labels = []
        for idx, cls in enumerate(self.classes):
            folder = os.path.join(root, cls)
            for img in os.listdir(folder):
                self.images.append(os.path.join(folder, img))
                self.labels.append(idx)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        img = test_transform(img)
        return img, self.labels[idx]


train_ds = FeatureDataset(TRAIN_PATH)
test_ds = FeatureDataset(TEST_PATH)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)

# -----------------------------------------------------
# FEATURE EXTRACTION (frozen encoder, same as CELL-4)
# -----------------------------------------------------

def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            f = ssl_model.encoder(x)
            feats.append(f.cpu())
            labels.append(y)
    return torch.cat(feats), torch.cat(labels)


train_features, train_labels = extract_features(train_loader)
test_features, test_labels = extract_features(test_loader)

print("Train:", train_features.shape)
print("Test :", test_features.shape)

FEAT_DIM = train_features.shape[1]

# -----------------------------------------------------
# 5-SHOT SUPPORT SET (identical construction to CELL-4)
# -----------------------------------------------------

N_SHOT = 5

support_feats = []
support_lbls = []

classes = torch.unique(train_labels)

for cls in classes:
    idx = torch.where(train_labels == cls)[0]
    idx = idx[torch.randperm(len(idx))[:N_SHOT]]
    support_feats.append(train_features[idx])
    support_lbls.append(train_labels[idx])

support_feats = torch.cat(support_feats).to(device)
support_lbls = torch.cat(support_lbls).to(device)

print("Support Samples:", len(support_lbls))

test_features_dev = test_features.to(device)
test_labels_np = test_labels.numpy()

# Remaining train samples (excluding support) used to TRAIN the lightweight
# comparison heads (Siamese / Relation) — the frozen encoder itself is never
# touched, only these small heads on top of it.
support_idx_global = []
for cls in classes:
    idx = torch.where(train_labels == cls)[0]
    idx = idx[torch.randperm(len(idx))[:N_SHOT]]
    support_idx_global.append(idx)
support_idx_global = torch.cat(support_idx_global)

all_idx = torch.arange(len(train_labels))
head_train_mask = torch.ones(len(train_labels), dtype=torch.bool)
head_train_mask[support_idx_global] = False

head_train_feats = train_features[head_train_mask].to(device)
head_train_lbls = train_labels[head_train_mask].to(device)


def make_pairs(feats, lbls, num_pairs=4000):
    """Sample balanced positive/negative pairs for training comparison heads."""
    n = len(lbls)
    a_idx = torch.randint(0, n, (num_pairs,))
    b_idx = torch.randint(0, n, (num_pairs,))
    labels_a = lbls[a_idx]
    labels_b = lbls[b_idx]
    pair_labels = (labels_a == labels_b).float()
    return feats[a_idx], feats[b_idx], pair_labels.to(device)


# =====================================================
# METHOD 1: SIAMESE NETWORK
# Twin embedding head trained with contrastive loss on pairs,
# then classify each query by nearest support (learned embedding space).
# =====================================================

class SiameseHead(nn.Module):
    def __init__(self, in_dim=FEAT_DIM, hidden_dim=256, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def contrastive_loss(emb_a, emb_b, pair_labels, margin=1.0):
    dist = F.pairwise_distance(emb_a, emb_b)
    pos_loss = pair_labels * dist.pow(2)
    neg_loss = (1 - pair_labels) * torch.clamp(margin - dist, min=0).pow(2)
    return (pos_loss + neg_loss).mean()


def train_siamese(epochs=30, num_pairs=4000, lr=1e-3):
    head = SiameseHead().to(device)
    optimizer = optim.Adam(head.parameters(), lr=lr)

    for epoch in range(epochs):
        feat_a, feat_b, pair_labels = make_pairs(head_train_feats, head_train_lbls, num_pairs)
        emb_a, emb_b = head(feat_a), head(feat_b)
        loss = contrastive_loss(emb_a, emb_b, pair_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"  [Siamese] epoch {epoch + 1}/{epochs} loss={loss.item():.4f}")

    return head


@torch.no_grad()
def predict_siamese(head):
    head.eval()
    support_emb = head(support_feats)
    query_emb = head(test_features_dev)

    preds = []
    for q in query_emb:
        dists = F.pairwise_distance(q.unsqueeze(0).expand(len(support_emb), -1), support_emb)
        per_class_dist = torch.stack([
            dists[support_lbls == c].mean() for c in classes
        ])
        preds.append(torch.argmin(per_class_dist).item())
    return np.array(preds)


print("\nTraining Siamese Network head...")
siamese_head = train_siamese()
siamese_preds = predict_siamese(siamese_head)


# =====================================================
# METHOD 2: MATCHING NETWORK
# Attention (softmax over cosine similarity to support set) -> weighted
# label vote, as in Vinyals et al. 2016 (simplified: no full-context
# embedding LSTM, standard for a frozen-encoder inference-stage swap).
# =====================================================

@torch.no_grad()
def predict_matching_network():
    support_norm = F.normalize(support_feats, dim=1)
    query_norm = F.normalize(test_features_dev, dim=1)

    # cosine similarity: [num_query, num_support]
    sim = torch.mm(query_norm, support_norm.t())
    attention = F.softmax(sim, dim=1)

    # one-hot support labels -> weighted class scores
    support_onehot = F.one_hot(support_lbls, num_classes=NUM_CLASSES).float()
    class_scores = torch.mm(attention, support_onehot)

    preds = torch.argmax(class_scores, dim=1)
    return preds.cpu().numpy()


print("\nRunning Matching Network (attention-based)...")
matching_preds = predict_matching_network()


# =====================================================
# METHOD 3: RELATION NETWORK
# Small relation module scores each (query, support) pair; trained via
# MSE regression to 1 (same class) / 0 (different class), as in
# Sung et al. 2018. Prediction = class with highest mean relation score.
# =====================================================

class RelationModule(nn.Module):
    def __init__(self, in_dim=FEAT_DIM, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim * 2, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid(),
        )

    def forward(self, feat_a, feat_b):
        x = torch.cat([feat_a, feat_b], dim=1)
        return self.net(x).squeeze(1)


def train_relation(epochs=30, num_pairs=4000, lr=1e-3):
    module = RelationModule().to(device)
    optimizer = optim.Adam(module.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        feat_a, feat_b, pair_labels = make_pairs(head_train_feats, head_train_lbls, num_pairs)
        scores = module(feat_a, feat_b)
        loss = criterion(scores, pair_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"  [Relation] epoch {epoch + 1}/{epochs} loss={loss.item():.4f}")

    return module


@torch.no_grad()
def predict_relation(module):
    module.eval()
    preds = []
    for q in test_features_dev:
        q_rep = q.unsqueeze(0).expand(len(support_feats), -1)
        scores = module(q_rep, support_feats)
        per_class_score = torch.stack([
            scores[support_lbls == c].mean() for c in classes
        ])
        preds.append(torch.argmax(per_class_score).item())
    return np.array(preds)


print("\nTraining Relation Network module...")
relation_module = train_relation()
relation_preds = predict_relation(relation_module)


# =====================================================
# METRICS + COMPARISON TABLE
# =====================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return acc, prec, rec, f1


results = {
    "Siamese Network": compute_metrics(test_labels_np, siamese_preds),
    "Matching Network": compute_metrics(test_labels_np, matching_preds),
    "Relation Network": compute_metrics(test_labels_np, relation_preds),
}

comparison_df = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
    }
    for name, (acc, prec, rec, f1) in results.items()
])

print("\n" + "=" * 60)
print("5-SHOT INFERENCE COMPARISON (same fine-tuned SSL encoder)")
print("=" * 60)
print(comparison_df.to_string(index=False))

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "fewshot_comparison.csv"), index=False)
print(f"\nSaved: {os.path.join(OUTPUT_DIR, 'fewshot_comparison.csv')}")

try:
    display(comparison_df)
except NameError:
    pass